# 🌊 Ocean Floor & Seabed Visualization
## 2D (ArcGIS) + 3D Interactive Surface (Plotly)

### Data Sources — All Free, No Login Required
| Source | Data | URL |
|---|---|---|
| **NOAA NCEI** | Global bathymetry DEM (GEBCO-derived) | `gis.ngdc.noaa.gov` ImageServer |
| **NOAA NCEI** | ETOPO1 Bedrock (ocean + land elevation) | `gis.ngdc.noaa.gov` ImageServer |
| **NOAA NCEI** | GEBCO 2019 depth contours | `gis.ngdc.noaa.gov` MapServer |
| **NOAA NCEI** | EMAG2v3 Magnetic anomaly | `gis.ngdc.noaa.gov` ImageServer |
| **NOAA NCEI** | Multibeam sonar survey coverage | `gis.ngdc.noaa.gov` MapServer |
| **InterRidge** | 721 hydrothermal vent fields (v3.4, 2020) | PANGAEA CSV |
| **Smithsonian GVP** | Submarine volcanoes KML | `volcano.si.edu` |
| **USGS** | Tectonic plate boundaries | `earthquake.usgs.gov` MapServer |

### What This Notebook Covers
1. 📦 Install & setup
2. 🗺️ 2D: Global bathymetry (depth map)
3. 🗺️ 2D: ETOPO1 ocean floor
4. 🗺️ 2D: GEBCO depth contours
5. 🗺️ 2D: Magnetic anomaly map (seafloor spreading)
6. 🗺️ 2D: Multibeam sonar survey coverage
7. 🔵 2D: Hydrothermal vent fields (InterRidge)
8. 🌋 2D: Submarine volcanoes (Smithsonian GVP)
9. 🗺️ 2D: Combined ocean floor dashboard
10. 📊 DataFrame: Query & analyze vent/volcano data
11. **🌍 3D: Interactive global ocean floor surface (Plotly)**
12. **🌍 3D: Regional deep focus (Mariana Trench, Mid-Atlantic Ridge, etc.)**
13. **🌍 3D: Overlay vents + volcanoes on 3D surface**
14. 📍 Focus on any ocean region
---

## 📦 Step 1 — Install & Import

In [1]:
!pip install arcgis arcgis-mapping plotly tifffile requests kaleido --quiet
print('✅ All packages installed')

✅ All packages installed


In [2]:
import io, json, zipfile, urllib.request
import numpy as np
import pandas as pd
import requests
import tifffile
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from arcgis.gis        import GIS
from arcgis.geocoding  import geocode
from arcgis.geometry   import Geometry
from arcgis.layers     import MapImageLayer
from arcgis.map.symbols import SimpleMarkerSymbolEsriSMS

gis = GIS()
print('✅ Connected to ArcGIS:', gis)

# ── All confirmed-working REST service URLs ───────────────────────────────────
NOAA   = 'https://gis.ngdc.noaa.gov/arcgis/rest/services'
USGS   = 'https://earthquake.usgs.gov/arcgis/rest/services'

URLS = {
    # Raster / Image services — bathymetry & geology
    'bathy_hillshade'  : f'{NOAA}/DEM_mosaics/DEM_global_mosaic_hillshade/ImageServer',
    'bathy_dem'        : f'{NOAA}/DEM_mosaics/DEM_global_mosaic/ImageServer',
    'etopo1'           : f'{NOAA}/DEM_mosaics/ETOPO1_bedrock/ImageServer',
    'etopo1_ice'       : f'{NOAA}/DEM_mosaics/ETOPO1_ice_surface/ImageServer',
    'magnetic_anomaly' : f'{NOAA}/EMAG2v3/ImageServer',
    # Vector / Map services
    'gebco_contours'   : f'{NOAA}/wgs84/gebco_2019_contours/MapServer',
    'multibeam'        : f'{NOAA}/wgs84/multibeam_footprints/MapServer',
    'coastlines'       : f'{NOAA}/reference/coastlines/MapServer',
    'plate_boundaries' : f'{USGS}/eq/map_plateboundaries/MapServer',
    'faults'           : f'{USGS}/eq/map_faults/MapServer',
    # Point data downloads
    'vents_csv'        : 'https://vents-data.interridge.org/ventfields/export.csv',
    'volcanoes_kml'    : 'https://volcano.si.edu/ge/GVPWorldVolcanoes.kml',
}

# ── NOAA ExportImage endpoint (for 3D data) ───────────────────────────────────
BATHY_EXPORT = f"{URLS['bathy_dem']}/exportImage"

print('\n📡 All data source URLs configured')
for k, v in URLS.items():
    print(f'   {k:<20} : {v[:60]}...' if len(v) > 60 else f'   {k:<20} : {v}')

✅ Connected to ArcGIS: GIS @ https://www.arcgis.com version:2026.1

📡 All data source URLs configured
   bathy_hillshade      : https://gis.ngdc.noaa.gov/arcgis/rest/services/DEM_mosaics/D...
   bathy_dem            : https://gis.ngdc.noaa.gov/arcgis/rest/services/DEM_mosaics/D...
   etopo1               : https://gis.ngdc.noaa.gov/arcgis/rest/services/DEM_mosaics/E...
   etopo1_ice           : https://gis.ngdc.noaa.gov/arcgis/rest/services/DEM_mosaics/E...
   magnetic_anomaly     : https://gis.ngdc.noaa.gov/arcgis/rest/services/EMAG2v3/Image...
   gebco_contours       : https://gis.ngdc.noaa.gov/arcgis/rest/services/wgs84/gebco_2...
   multibeam            : https://gis.ngdc.noaa.gov/arcgis/rest/services/wgs84/multibe...
   coastlines           : https://gis.ngdc.noaa.gov/arcgis/rest/services/reference/coa...
   plate_boundaries     : https://earthquake.usgs.gov/arcgis/rest/services/eq/map_plat...
   faults               : https://earthquake.usgs.gov/arcgis/rest/services/eq/map_faul..

## 🔧 Step 2 — Helper Functions

In [4]:
# ── Safe REST loader ──────────────────────────────────────────────────────────
def safe_load(url, name):
    try:
        lyr = MapImageLayer(url)
        _   = lyr.properties
        print(f'  ✅ {name}')
        return lyr
    except Exception as e:
        print(f'  ❌ {name} — {str(e)[:60]}')
        return None

# ── Map factory ───────────────────────────────────────────────────────────────
def make_map(location='World', basemap='oceans', zoom=2, height='580px'):
    m = gis.map(location)
    m.basemap.basemap = basemap
    m.zoom            = zoom
    m.layout.height   = height
    return m

# ── Point draw symbol ─────────────────────────────────────────────────────────
def make_symbol(color, size=10, style='esriSMSCircle'):
    return SimpleMarkerSymbolEsriSMS(**{
        'type': 'esriSMS', 'style': style,
        'color': color, 'size': size,
        'angle': 0, 'xoffset': 0, 'yoffset': 0,
        'outline': {'type': 'esriSLS', 'style': 'esriSLSSolid',
                    'color': [255, 255, 255, 120], 'width': 0.5}
    })

def draw_points(m, df, lat_col, lon_col, color, size=8, label='points'):
    drawn = 0
    for _, row in df.dropna(subset=[lat_col, lon_col]).iterrows():
        pt = Geometry({'x': float(row[lon_col]), 'y': float(row[lat_col]),
                       'spatialReference': {'wkid': 4326}})
        m.content.draw(pt, symbol=make_symbol(color, size))
        drawn += 1
    print(f'  ✅ Drew {drawn} {label}')

# ── Download bathymetry as numpy array ────────────────────────────────────────
def fetch_bathymetry(bbox=(-180, -90, 180, 90), size=(720, 360), label='Global'):
    """
    Download bathymetry/elevation grid from NOAA NCEI as float32 numpy array.
    bbox: (xmin, ymin, xmax, ymax) in WGS84
    size: (width, height) in pixels
    Returns: numpy array of elevation values (negative = below sea level)
    """
    print(f'📡 Downloading {label} bathymetry ({size[0]}×{size[1]} grid)...')
    params = {
        'bbox'                   : f'{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}',
        'bboxSR'                 : 4326,
        'size'                   : f'{size[0]},{size[1]}',
        'imageSR'                : 4326,
        'format'                 : 'tiff',
        'pixelType'              : 'F32',
        'noData'                 : '',
        'noDataInterpretation'   : 'esriNoDataMatchAny',
        'interpolation'          : 'RSP_BilinearInterpolation',
        'renderingRule'          : '',
        'f'                      : 'image',
    }
    resp = requests.get(BATHY_EXPORT, params=params, timeout=120)
    resp.raise_for_status()
    arr = tifffile.imread(io.BytesIO(resp.content))
    print(f'  ✅ Downloaded: shape={arr.shape}, min={arr.min():.0f}m, max={arr.max():.0f}m')
    return arr

# ── Ocean colorscale ──────────────────────────────────────────────────────────
OCEAN_COLORSCALE = [
    [0.0,  '#08003f'],   # > -11000m abyssal (darkest)
    [0.15, '#0a1a6e'],   # -8000m
    [0.30, '#0d3b9e'],   # -6000m
    [0.45, '#1565c0'],   # -4000m mid-ocean ridge flank
    [0.60, '#1e88e5'],   # -2000m
    [0.72, '#42a5f5'],   # -1000m
    [0.82, '#90caf9'],   # -200m  continental shelf
    [0.90, '#bbdefb'],   # -50m
    [0.94, '#e3f2fd'],   # 0m sea level
    [0.96, '#a5d6a7'],   # +500m land
    [1.0,  '#4e342e'],   # +8000m mountain peak
]

print('✅ All helpers ready')

✅ All helpers ready


---
## 🗺️ Step 3 — 2D: Global Bathymetry (NOAA NCEI Hillshade)
The definitive ocean depth map — GEBCO-derived, full global coverage, 1 arc-second resolution.

In [5]:
bathy_map = make_map(basemap='oceans', zoom=2)

print('Loading NOAA bathymetry layers...')
hillshade = safe_load(URLS['bathy_hillshade'], 'Global Bathymetry Hillshade (NOAA NCEI)')
coastline = safe_load(URLS['coastlines'],      'Coastlines reference')

# Add deepest layer first (hillshade), then coastlines on top
for lyr in [hillshade, coastline]:
    if lyr: bathy_map.content.add(lyr)

print('\n🌊 Color guide:')
print('   Deep purple/black = deepest ocean (>8,000m below sea level)')
print('   Dark blue         = abyssal plains (4,000–6,000m)')
print('   Medium blue       = mid-ocean ridges & rises')
print('   Light blue        = continental shelves (0–200m)')
bathy_map

Loading NOAA bathymetry layers...
  ✅ Global Bathymetry Hillshade (NOAA NCEI)
  ✅ Coastlines reference

🌊 Color guide:
   Deep purple/black = deepest ocean (>8,000m below sea level)
   Dark blue         = abyssal plains (4,000–6,000m)
   Medium blue       = mid-ocean ridges & rises
   Light blue        = continental shelves (0–200m)


Map(layout=Layout(height='580px'))

---
## 🗺️ Step 4 — 2D: ETOPO1 Ocean Floor
ETOPO1 is a 1 arc-minute global relief model integrating ocean bathymetry and land topography.  
Shows the full Earth surface including ocean ridges, trenches, and continental margins.

In [6]:
etopo_map = make_map(basemap='oceans', zoom=2)

print('Loading ETOPO1 layers...')
etopo = safe_load(URLS['etopo1'], 'ETOPO1 Bedrock (ocean + land elevation)')

if etopo: etopo_map.content.add(etopo)

print('\n📍 Features visible in ETOPO1:')
print('   • Mid-Ocean Ridges   — elevated submarine mountain chains')
print('   • Abyssal Plains     — flat deep-ocean floor')
print('   • Ocean Trenches     — deepest points on Earth')
print('   • Continental Shelf  — shallow areas around continents')
print('   • Seamounts          — isolated underwater mountains')
etopo_map

Loading ETOPO1 layers...
  ✅ ETOPO1 Bedrock (ocean + land elevation)

📍 Features visible in ETOPO1:
   • Mid-Ocean Ridges   — elevated submarine mountain chains
   • Abyssal Plains     — flat deep-ocean floor
   • Ocean Trenches     — deepest points on Earth
   • Continental Shelf  — shallow areas around continents
   • Seamounts          — isolated underwater mountains


Map(layout=Layout(height='580px'))

---
## 🗺️ Step 5 — 2D: GEBCO Depth Contours
Contour lines at standard ocean depth intervals — shows the shape of the ocean floor.

In [7]:
gebco_map = make_map(basemap='oceans', zoom=2)

print('Loading GEBCO layers...')
gebco = safe_load(URLS['gebco_contours'], 'GEBCO 2019 Depth Contours')
plates = safe_load(URLS['plate_boundaries'], 'Tectonic Plate Boundaries')

for lyr in [gebco, plates]:
    if lyr: gebco_map.content.add(lyr)

print('\n📏 Contour interval key:')
print('   200m, 500m, 1000m, 2000m, 3000m, 4000m, 5000m, 6000m, 7000m+')
gebco_map

Loading GEBCO layers...
  ✅ GEBCO 2019 Depth Contours
  ✅ Tectonic Plate Boundaries

📏 Contour interval key:
   200m, 500m, 1000m, 2000m, 3000m, 4000m, 5000m, 6000m, 7000m+


Map(layout=Layout(height='580px'))

---
## 🗺️ Step 6 — 2D: Magnetic Anomaly Map (EMAG2v3)
Seafloor magnetic anomalies reveal **seafloor spreading** — where new ocean crust is created at mid-ocean ridges.  
Symmetric striped patterns on either side of a ridge = different ages of oceanic crust.

In [8]:
mag_map = make_map(basemap='oceans', zoom=2)

print('Loading magnetic anomaly layer...')
emag = safe_load(URLS['magnetic_anomaly'], 'EMAG2v3 Magnetic Anomaly')

if emag: mag_map.content.add(emag)

print('\n🔴🔵 Color guide (nanotesla scale):')
print('   Red/Warm  = positive anomaly (young oceanic crust near ridge)')
print('   Blue/Cool = negative anomaly (reversed polarity zones)')
print('   Striped patterns along mid-ocean ridges = seafloor spreading record')
mag_map

Loading magnetic anomaly layer...
  ✅ EMAG2v3 Magnetic Anomaly

🔴🔵 Color guide (nanotesla scale):
   Red/Warm  = positive anomaly (young oceanic crust near ridge)
   Blue/Cool = negative anomaly (reversed polarity zones)
   Striped patterns along mid-ocean ridges = seafloor spreading record


Map(layout=Layout(height='580px'))

---
## 🗺️ Step 7 — 2D: Multibeam Sonar Survey Coverage
Shows where detailed high-resolution seafloor mapping has been done by research vessels.  
Less than 25% of the ocean floor has been mapped this way — most is still unmapped.

In [9]:
# ── Two confirmed-working NOAA multibeam URLs ─────────────────────────────────
MULTIBEAM_MAP    = 'https://gis.ngdc.noaa.gov/arcgis/rest/services/web_mercator/multibeam_dynamic/MapServer'
MULTIBEAM_MOSAIC = 'https://gis.ngdc.noaa.gov/arcgis/rest/services/multibeam_mosaic/ImageServer'

sonar_map = make_map(basemap='hybrid', zoom=2)

print('Loading multibeam sonar layers...')

# Option A: Survey footprints (polygon outlines of each research cruise track)
multibeam_surveys = safe_load(MULTIBEAM_MAP,    'Multibeam Survey Footprints (dynamic MapServer)')

# Option B: Actual bathymetry mosaic from all multibeam surveys merged
multibeam_mosaic  = safe_load(MULTIBEAM_MOSAIC, 'Multibeam Bathymetry Mosaic (ImageServer)')

# Add mosaic first (base layer), then survey outlines on top
for lyr in [multibeam_mosaic, multibeam_surveys]:
    if lyr: sonar_map.content.add(lyr)

print('\n🔊 What you are seeing:')
print('   Colored polygons = individual research vessel survey tracks')
print('   Color = survey year (green=1980s → yellow=1990s → orange=2000s → red=2020s)')
print('   Underlying mosaic = merged bathymetry from all surveys')
print('   Empty ocean areas = still unmapped at high resolution as of 2024')
sonar_map

Loading multibeam sonar layers...
  ✅ Multibeam Survey Footprints (dynamic MapServer)
  ✅ Multibeam Bathymetry Mosaic (ImageServer)

🔊 What you are seeing:
   Colored polygons = individual research vessel survey tracks
   Color = survey year (green=1980s → yellow=1990s → orange=2000s → red=2020s)
   Underlying mosaic = merged bathymetry from all surveys
   Empty ocean areas = still unmapped at high resolution as of 2024


Map(layout=Layout(height='580px'))

---
## 🔵 Step 8 — 2D: Hydrothermal Vent Fields (InterRidge v3.4)
721 known hydrothermal vent fields worldwide — the official global database from InterRidge.  
All located along mid-ocean ridges and subduction zones.

In [10]:
print('📡 Downloading InterRidge hydrothermal vents database...')

vents_df = None
try:
    # InterRidge public CSV export
    resp = requests.get(URLS['vents_csv'], timeout=30)
    resp.raise_for_status()
    vents_df = pd.read_csv(io.StringIO(resp.text))
    print(f'✅ Loaded {len(vents_df)} vent fields from InterRidge')
    print('Columns:', vents_df.columns.tolist())
except Exception as e:
    print(f'⚠️ InterRidge live API unavailable ({e})')
    print('   Using PANGAEA DOI backup...')
    try:
        # PANGAEA backup — flat tab-separated download
        pangaea_url = 'https://doi.pangaea.de/10.1594/PANGAEA.917894?format=textfile'
        resp = requests.get(pangaea_url, timeout=30)
        resp.raise_for_status()
        # PANGAEA text files have metadata header lines starting with '*'
        lines = [l for l in resp.text.split('\n') if not l.startswith('*')]
        vents_df = pd.read_csv(io.StringIO('\n'.join(lines)), sep='\t')
        print(f'✅ Loaded {len(vents_df)} vent fields from PANGAEA backup')
        print('Columns:', vents_df.columns.tolist())
    except Exception as e2:
        print(f'❌ Both sources failed: {e2}')
        print('   Creating sample dataset with known major vent fields...')
        # Hardcoded fallback with real major vent fields
        vents_df = pd.DataFrame({
            'Name'          : ['Lost City','TAG','Rainbow','Lucky Strike','Broken Spur',
                               'Main Endeavour','Juan de Fuca','EPR 9N','EPR 21N',
                               'Kairei','Logatchev','Ashadze','PACMANUS','Manus Basin',
                               'Brothers','Lau Basin','Central Indian Ridge','SWIR'],
            'Latitude'      : [30.12,-26.14,36.23,37.29,29.17,47.95,45.93,9.83,21.0,
                                -25.33,14.72,12.97,-3.72,-3.72,-34.87,-20.0,-5.0,-37.0],
            'Longitude'     : [-42.12,-44.83,-33.90,-32.28,-44.17,-129.10,-130.0,-104.29,-109.05,
                                 70.04,-44.98,-44.92,151.67,151.67,-179.07,-176.0,67.0,50.0],
            'MaxDepth_m'    : [750,3620,2300,1700,3050,2200,2250,2500,2600,
                                2450,3020,4200,1680,1680,1200,1700,3300,2800],
            'Activity'      : ['active']*18,
        })
        print(f'✅ Fallback dataset: {len(vents_df)} known major vent fields')

if vents_df is not None:
    vents_df.head()

📡 Downloading InterRidge hydrothermal vents database...
⚠️ InterRidge live API unavailable (('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)))
   Using PANGAEA DOI backup...
❌ Both sources failed: Error tokenizing data. C error: Expected 2 fields in line 15, saw 4

   Creating sample dataset with known major vent fields...
✅ Fallback dataset: 18 known major vent fields


In [11]:
import io, requests
import pandas as pd

print('📡 Fetching InterRidge v3.4 from PANGAEA (official archive)...')

vents_df = None

# ── Primary source: PANGAEA tab-separated flat file ───────────────────────────
# Official DOI: https://doi.org/10.1594/PANGAEA.917894
PANGAEA_URL = 'https://doi.pangaea.de/10.1594/PANGAEA.917894?format=textfile'

try:
    resp = requests.get(PANGAEA_URL, timeout=30)
    resp.raise_for_status()

    # PANGAEA text files: metadata lines start with '*', data is tab-separated after them
    lines = resp.text.split('\n')
    data_lines = [l for l in lines if not l.startswith('*') and l.strip() != '']
    vents_df = pd.read_csv(io.StringIO('\n'.join(data_lines)), sep='\t')

    print(f'✅ Loaded from PANGAEA: {len(vents_df)} rows')
    print('   Raw columns:', vents_df.columns.tolist())

except Exception as e:
    print(f'⚠️ PANGAEA fetch failed: {e}')

# ── Fallback: hardcoded dataset of all major known vent fields ────────────────
if vents_df is None or len(vents_df) == 0 or 'Latitude' not in str(vents_df.columns):
    print('\n📋 Using hardcoded dataset (all major confirmed vent fields)...')
    vents_df = pd.DataFrame([
        # Mid-Atlantic Ridge
        ('Lost City',        30.12, -42.12, 750,  'active', 'MAR'),
        ('TAG',             -26.14, -44.83, 3620, 'active', 'MAR'),
        ('Rainbow',          36.23, -33.90, 2300, 'active', 'MAR'),
        ('Lucky Strike',     37.29, -32.28, 1700, 'active', 'MAR'),
        ('Broken Spur',      29.17, -43.17, 3050, 'active', 'MAR'),
        ('Logatchev',        14.72, -44.98, 3020, 'active', 'MAR'),
        ('Ashadze',          12.97, -44.92, 4200, 'active', 'MAR'),
        ('Snake Pit',        23.37, -44.95, 3450, 'active', 'MAR'),
        ('Menez Gwen',       37.84, -31.52, 850,  'active', 'MAR'),
        # East Pacific Rise
        ('EPR 9°N',           9.83,-104.29, 2500, 'active', 'EPR'),
        ('EPR 21°N',         21.00,-109.05, 2600, 'active', 'EPR'),
        ('EPR 13°N',         13.00,-103.93, 2630, 'active', 'EPR'),
        ('EPR 11°N',         11.00,-103.77, 2540, 'active', 'EPR'),
        ('EPR 18-19°S',     -18.50,-113.40, 2630, 'active', 'EPR'),
        # Juan de Fuca Ridge
        ('Main Endeavour',   47.95,-129.10, 2200, 'active', 'JdFR'),
        ('Axial Seamount',   45.93,-130.00, 1520, 'active', 'JdFR'),
        ('High Rise',        47.97,-129.09, 2205, 'active', 'JdFR'),
        ('Mothra',           47.92,-129.11, 2345, 'active', 'JdFR'),
        ('CoAxial',          46.50,-129.58, 2650, 'active', 'JdFR'),
        # Indian Ocean
        ('Kairei',          -25.33,  70.04, 2450, 'active', 'CIR'),
        ('Edmond',          -23.87,  69.60, 3290, 'active', 'CIR'),
        ('Longqi (Dragon)', -37.78,  49.65, 2750, 'active', 'SWIR'),
        ('Tiancheng',       -37.58,  50.46, 2870, 'active', 'SWIR'),
        ('Duanqiao',        -37.67,  50.47, 2695, 'active', 'SWIR'),
        # Western Pacific / Back-Arc
        ('PACMANUS',         -3.72, 151.67, 1680, 'active', 'Manus'),
        ('SuSu Knolls',      -3.80, 152.10, 1150, 'active', 'Manus'),
        ('Brothers',        -34.87,-179.07, 1200, 'active', 'Kermadec'),
        ('Rumble III',      -35.75,-178.48,  700, 'active', 'Kermadec'),
        ('Lau Basin',       -20.00,-176.00, 1700, 'active', 'Lau'),
        ('NW Rota-1',        14.60, 144.78, 555,  'active', 'Mariana'),
        ('Forecast',         13.37, 143.93, 1080, 'active', 'Mariana'),
        ('Champagne',        13.47, 144.04,  65,  'active', 'Mariana'),
        # Arctic
        ('Loki Castle',      73.77,  -8.16, 2350, 'active', 'AMOR'),
        ('Jan Mayen',        71.00,  -5.70, 550,  'active', 'AMOR'),
        ('Soria Moria',      73.67,  -7.00, 2000, 'active', 'AMOR'),
        # Other notable
        ('Guaymas Basin',    27.00,-111.40, 2000, 'active', 'Gulf of California'),
        ('Okinawa Trough',   27.50, 127.00, 700,  'active', 'Okinawa'),
        ('Myojin Knoll',     32.10, 139.90, 1000, 'active', 'Izu-Bonin'),
        ('Kick-em Jenny',    12.30, -61.64, 180,  'active', 'Caribbean'),
        ('Kolumbo',          36.52,  25.48, 500,  'active', 'Aegean'),
        ('Marsili',          39.27,  14.33, 450,  'active', 'Tyrrhenian'),
        ('Tagoro',           27.62, -18.03, 350,  'active', 'Canary'),
        ('West Mata',       -15.10,-173.03, 1174, 'active', 'Tonga'),
    ], columns=['name','lat','lon','depth_m','activity','region'])
    print(f'✅ Hardcoded fallback: {len(vents_df)} major confirmed vent fields')

# ── Standardize to known column names ─────────────────────────────────────────
# Handle PANGAEA column names: 'Latitude', 'Longitude', 'Vent field name', 'Maximum or single reported depth (m)'
if 'Latitude' in vents_df.columns:
    vents_df = vents_df.rename(columns={
        'Latitude'                               : 'lat',
        'Longitude'                              : 'lon',
        'Vent field name'                        : 'name',
        'Maximum or single reported depth (m)'   : 'depth_m',
        'Activity'                               : 'activity',
        'Tectonic setting'                       : 'region',
    })

# Ensure numeric, drop bad rows
vents_df['lat'] = pd.to_numeric(vents_df['lat'], errors='coerce')
vents_df['lon'] = pd.to_numeric(vents_df['lon'], errors='coerce')
vents_df = vents_df.dropna(subset=['lat','lon']).reset_index(drop=True)

show = [c for c in ['name','lat','lon','depth_m','activity','region'] if c in vents_df.columns]
print(f'\n✅ {len(vents_df)} vent fields ready')
vents_df[show].head(10)

📡 Fetching InterRidge v3.4 from PANGAEA (official archive)...
⚠️ PANGAEA fetch failed: Error tokenizing data. C error: Expected 2 fields in line 15, saw 4


📋 Using hardcoded dataset (all major confirmed vent fields)...
✅ Hardcoded fallback: 43 major confirmed vent fields

✅ 43 vent fields ready


,name,lat,lon,depth_m,activity,region
0,Lost City,30.12,-42.12,750,active,MAR
1,TAG,-26.14,-44.83,3620,active,MAR
2,Rainbow,36.23,-33.90,2300,active,MAR
3,Lucky Strike,37.29,-32.28,1700,active,MAR
4,Broken Spur,29.17,-43.17,3050,active,MAR
5,Logatchev,14.72,-44.98,3020,active,MAR
6,Ashadze,12.97,-44.92,4200,active,MAR
7,Snake Pit,23.37,-44.95,3450,active,MAR
8,Menez Gwen,37.84,-31.52,850,active,MAR
9,EPR 9°N,9.83,-104.29,2500,active,EPR


In [12]:
vent_map = make_map(basemap='hybrid', zoom=2)

# Add bathymetry base layer
bathy = safe_load(URLS['bathy_hillshade'], 'Bathymetry base')
plates = safe_load(URLS['plate_boundaries'], 'Plate boundaries')
for lyr in [bathy, plates]:
    if lyr: vent_map.content.add(lyr)

# Draw vent fields as cyan dots
if vents_df is not None:
    draw_points(vent_map, vents_df, 'lat', 'lon',
                color=[0, 230, 220, 240], size=9, label='hydrothermal vent fields')

print('\n🔵 Cyan dots = hydrothermal vent fields')
print('   Most cluster along mid-ocean ridges (plate spreading centers)')
print('   Deepest vents: >4,000m on slow-spreading ridges')
vent_map

  ✅ Bathymetry base
  ✅ Plate boundaries
  ✅ Drew 43 hydrothermal vent fields

🔵 Cyan dots = hydrothermal vent fields
   Most cluster along mid-ocean ridges (plate spreading centers)
   Deepest vents: >4,000m on slow-spreading ridges


Map(layout=Layout(height='580px'))

---
## 🌋 Step 9 — 2D: Submarine Volcanoes (Smithsonian GVP)
Submarine volcanoes from the Global Volcanism Program — all volcanoes with seafloor or near-shore activity.

In [13]:
import xml.etree.ElementTree as ET
import io, requests
import pandas as pd

print('📡 Downloading Smithsonian GVP volcano data...')

volc_df = None

# ── Try GVP CSV first (more reliable than KML) ────────────────────────────────
GVP_CSV = 'https://volcano.si.edu/database/search_eruption_excel.cfm'
GVP_TSV = 'https://volcano.si.edu/volcanolist_csv.cfm'

try:
    resp = requests.get(GVP_TSV, timeout=30, headers={'User-Agent': 'Mozilla/5.0'})
    resp.raise_for_status()
    volc_df = pd.read_csv(io.StringIO(resp.text), skiprows=1)
    print(f'✅ GVP CSV loaded: {len(volc_df)} rows')
    print('   Columns:', volc_df.columns.tolist())
except Exception as e:
    print(f'⚠️ GVP CSV failed: {e}')

# ── Try KML with deeper search if CSV failed ──────────────────────────────────
if volc_df is None or len(volc_df) == 0:
    try:
        resp = requests.get('https://volcano.si.edu/ge/GVPWorldVolcanoes.kml',
                            timeout=30, headers={'User-Agent': 'Mozilla/5.0'})
        resp.raise_for_status()
        root = ET.fromstring(resp.content)

        # Search ALL tags — GVP KML may use non-standard nesting
        rows = []
        all_tags = set(el.tag for el in root.iter())
        print('   KML tags found:', [t.split('}')[-1] for t in list(all_tags)[:15]])

        # Try every namespace variant
        for ns_uri in ['http://www.opengis.net/kml/2.2',
                       'http://earth.google.com/kml/2.1',
                       'http://earth.google.com/kml/2.2', '']:
            tag = f'{{{ns_uri}}}Placemark' if ns_uri else 'Placemark'
            placemarks = root.findall(f'.//{tag}')
            if placemarks:
                print(f'   Found {len(placemarks)} Placemarks with ns="{ns_uri}"')
                for pm in placemarks:
                    def ft(tag_name):
                        for ns in [ns_uri, '']:
                            t = f'{{{ns}}}{tag_name}' if ns else tag_name
                            v = pm.findtext(f'.//{t}')
                            if v: return v
                        return None
                    name   = ft('name') or 'Unknown'
                    coords = ft('coordinates')
                    if coords:
                        parts = coords.strip().split(',')
                        if len(parts) >= 2:
                            try:
                                rows.append({'name': name,
                                             'lon': float(parts[0]),
                                             'lat': float(parts[1])})
                            except ValueError:
                                pass
                break  # stop after first working namespace

        if rows:
            volc_df = pd.DataFrame(rows)
            print(f'✅ Parsed {len(volc_df)} volcanoes from KML')
        else:
            print('   KML parsed 0 rows — using hardcoded fallback')
    except Exception as e:
        print(f'⚠️ KML also failed: {e}')

# ── Hardcoded fallback — all major submarine/subduction volcanoes ─────────────
if volc_df is None or len(volc_df) == 0:
    print('📋 Using hardcoded dataset...')
    volc_df = pd.DataFrame([
        # Pacific Ring of Fire
        ('Kavachi',            -8.99,  157.98, 'Solomon Islands',    'submarine'),
        ('NW Rota-1',          14.60,  144.78, 'Mariana Arc',        'submarine'),
        ('Monowai',           -25.89, -177.19, 'Kermadec Arc',       'submarine'),
        ('West Mata',         -15.10, -173.03, 'Tonga Arc',          'submarine'),
        ('Havre',             -31.73, -179.04, 'Kermadec Arc',       'submarine'),
        ('Brothers',          -34.87, -179.07, 'Kermadec Arc',       'submarine'),
        ('Rumble III',        -35.74, -178.48, 'Kermadec Arc',       'submarine'),
        ('Rumble V',          -36.14, -178.20, 'Kermadec Arc',       'submarine'),
        ('Axial Seamount',     45.95, -130.02, 'Juan de Fuca',       'submarine'),
        ('Brimstone Pit',      16.76,  -92.66, 'Caribbean',          'submarine'),
        ('Vailulu\'u',        -14.22, -169.03, 'Samoa',              'submarine'),
        ('Nafanua',           -14.22, -169.03, 'Samoa',              'submarine'),
        ('Lō\'ihi',           -18.92, -155.27, 'Hawaii',             'submarine'),
        ('Myojin Knoll',       32.10,  139.90, 'Izu-Bonin',          'submarine'),
        ('Nikko',              23.09,  142.35, 'Mariana Arc',        'submarine'),
        ('Fukutoku-Oka-no-Ba', 24.28,  141.48, 'Izu-Bonin',          'submarine'),
        ('Ahyi',               20.43,  145.03, 'Mariana Arc',        'submarine'),
        ('Daikoku',            21.32,  144.19, 'Mariana Arc',        'submarine'),
        ('Zealandia Bank',     13.99,  144.65, 'Mariana Arc',        'submarine'),
        # Atlantic
        ('Kick-em Jenny',      12.30,  -61.64, 'Caribbean',          'submarine'),
        ('Kolumbo',            36.52,   25.48, 'Aegean',             'submarine'),
        ('Marsili',            39.27,   14.33, 'Tyrrhenian',         'submarine'),
        ('Tagoro',             27.62,  -18.03, 'Canary Islands',     'submarine'),
        ('Fonualei',          -18.02, -174.32, 'Tonga',              'submarine'),
        # Indian Ocean
        ('Barren Island',      12.28,   93.86, 'Andaman',            'subaerial'),
        ('Fani Maoré',        -12.80,   45.50, 'Mayotte',            'submarine'),
        # Southern Ocean / Antarctic
        ('Orca',              -60.20,  -25.40, 'South Sandwich',     'submarine'),
        ('Protector Shoal',   -55.92,  -28.09, 'South Sandwich',     'submarine'),
    ], columns=['name','lat','lon','region','type'])
    print(f'✅ Hardcoded: {len(volc_df)} submarine/arc volcanoes')

# ── Standardize ───────────────────────────────────────────────────────────────
# Handle GVP CSV column names if that source worked
if 'Latitude' in volc_df.columns:
    volc_df = volc_df.rename(columns={
        'Volcano Name': 'name',
        'Latitude'    : 'lat',
        'Longitude'   : 'lon',
        'Region'      : 'region',
        'Primary Volcano Type': 'type',
    })

volc_df['lat'] = pd.to_numeric(volc_df['lat'], errors='coerce')
volc_df['lon'] = pd.to_numeric(volc_df['lon'], errors='coerce')
volc_df = volc_df.dropna(subset=['lat','lon']).reset_index(drop=True)

show = [c for c in ['name','lat','lon','region','type'] if c in volc_df.columns]
print(f'\n✅ {len(volc_df)} volcanoes ready')
volc_df[show].head(10)

📡 Downloading Smithsonian GVP volcano data...
⚠️ GVP CSV failed: No columns to parse from file
   KML tags found: ['refreshMode', 'Link', 'kml', 'Snippet', 'open', 'NetworkLink', 'name', 'refreshInterval', 'description', 'href']
   KML parsed 0 rows — using hardcoded fallback
📋 Using hardcoded dataset...
✅ Hardcoded: 28 submarine/arc volcanoes

✅ 28 volcanoes ready


,name,lat,lon,region,type
0,Kavachi,-8.99,157.98,Solomon Islands,submarine
1,NW Rota-1,14.60,144.78,Mariana Arc,submarine
2,Monowai,-25.89,-177.19,Kermadec Arc,submarine
3,West Mata,-15.10,-173.03,Tonga Arc,submarine
4,Havre,-31.73,-179.04,Kermadec Arc,submarine
5,Brothers,-34.87,-179.07,Kermadec Arc,submarine
6,Rumble III,-35.74,-178.48,Kermadec Arc,submarine
7,Rumble V,-36.14,-178.20,Kermadec Arc,submarine
8,Axial Seamount,45.95,-130.02,Juan de Fuca,submarine
9,Brimstone Pit,16.76,-92.66,Caribbean,submarine


In [14]:
# ── Fixed draw_points — guards against empty DataFrames ──────────────────────
def draw_points(m, df, lat_col, lon_col, color, size=8, label='points'):
    """Draw points on a Map widget. Safe against empty or missing-column DataFrames."""
    if df is None or len(df) == 0:
        print(f'  ⚠️  Skipped {label} — empty dataset')
        return
    if lat_col not in df.columns or lon_col not in df.columns:
        print(f'  ⚠️  Skipped {label} — columns {lat_col!r}/{lon_col!r} not in {df.columns.tolist()}')
        return
    valid = df.dropna(subset=[lat_col, lon_col])
    for _, row in valid.iterrows():
        pt = Geometry({'x': float(row[lon_col]), 'y': float(row[lat_col]),
                       'spatialReference': {'wkid': 4326}})
        m.content.draw(pt, symbol=make_symbol(color, size))
    print(f'  ✅ Drew {len(valid)} {label}')

In [15]:
# ── Now draw the map ──────────────────────────────────────────────────────────
volc_map = make_map(basemap='hybrid', zoom=2)

bathy = safe_load(URLS['bathy_hillshade'], 'Bathymetry base')
if bathy: volc_map.content.add(bathy)

draw_points(volc_map, volc_df,  'lat', 'lon', color=[255, 100, 0, 240], size=10, label='submarine volcanoes')
draw_points(volc_map, vents_df, 'lat', 'lon', color=[0, 230, 220, 200], size=7,  label='hydrothermal vents')

print('\n🟠 Orange = submarine volcanoes  |  🔵 Cyan = hydrothermal vents')
volc_map

  ✅ Bathymetry base
  ✅ Drew 28 submarine volcanoes
  ✅ Drew 43 hydrothermal vents

🟠 Orange = submarine volcanoes  |  🔵 Cyan = hydrothermal vents


Map(layout=Layout(height='580px'))

---
## 🗺️ Step 10 — 2D: Combined Ocean Floor Dashboard
All layers stacked in one map.

In [17]:
# ── Updated URLS dict — replace the two broken entries ────────────────────────
# Remove these from your URLS dict (both are dead):
#   'faults'    : f'{USGS}/eq/map_faults/MapServer'              ← 500 intermittent
#   'multibeam' : f'{NOAA}/wgs84/multibeam_footprints/MapServer' ← 404 permanently

# Add these confirmed-working replacements (from NOAA service directory):
URLS['multibeam']      = f'{NOAA}/web_mercator/multibeam_dynamic/MapServer'
URLS['marine_geology'] = f'{NOAA}/web_mercator/marine_geology_dynamic/MapServer'
URLS['hazards']        = f'{NOAA}/web_mercator/hazards/MapServer'
# hazards contains: tsunami events, significant earthquakes, significant volcanic
# eruptions, and Smithsonian volcano locations — replaces the faults layer

print('✅ URLs updated')

✅ URLs updated


In [ ]:
combo_map = make_map(basemap='hybrid', zoom=2, height='650px')

print('Loading all layers...')
layer_order = [
    (URLS['bathy_hillshade'],  'Bathymetry hillshade'),
    (URLS['gebco_contours'],   'GEBCO depth contours'),
    (URLS['plate_boundaries'], 'Tectonic plate boundaries'),   # USGS — usually up
    (URLS['marine_geology'],   'Marine geology surveys'),      # replaces faults ✅
    (URLS['hazards'],          'Hazards (tsunamis, volcanoes, earthquakes)'), # ✅
    (URLS['multibeam'],        'Multibeam sonar coverage'),    # corrected URL ✅
]

for url, name in layer_order:
    lyr = safe_load(url, name)
    if lyr: combo_map.content.add(lyr)

# Draw point features on top
draw_points(combo_map, volc_df,  'lat', 'lon', color=[255, 80, 0, 240],  size=11, label='submarine volcanoes')
draw_points(combo_map, vents_df, 'lat', 'lon', color=[0, 220, 210, 200], size=8,  label='hydrothermal vents')

print('\n✅ Combined ocean floor map ready!')
print('   🔵 Cyan          = hydrothermal vent fields')
print('   🟠 Orange        = submarine volcanoes')
print('   Plate boundaries = tectonic plate edges')
print('   Hazards layer    = tsunamis, significant quakes, GVP volcanoes')
print('   Marine geology   = research vessel survey tracks')
print('   Multibeam        = sonar coverage footprints')
combo_map

Loading all layers...
  ✅ Bathymetry hillshade
  ✅ GEBCO depth contours
  ✅ Tectonic plate boundaries
  ✅ Marine geology surveys
  ✅ Hazards (tsunamis, volcanoes, earthquakes)
  ✅ Multibeam sonar coverage
  ✅ Drew 28 submarine volcanoes


ConnectionError: A connection error has occurred: HTTPSConnectionPool(host='gis.ngdc.noaa.gov', port=443): Max retries exceeded with url: /arcgis/rest/services/web_mercator/multibeam_dynamic/MapServer (Caused by NameResolutionError("HTTPSConnection(host='gis.ngdc.noaa.gov', port=443): Failed to resolve 'gis.ngdc.noaa.gov' ([Errno 11001] getaddrinfo failed)"))

: 

---
## 📊 Step 11 — Analyze Vent & Volcano Data

In [ ]:
if vents_df is not None:
    print('🔵 Hydrothermal Vent Statistics:')
    print(f'   Total vent fields  : {len(vents_df)}')
    if 'depth_m' in vents_df.columns:
        depths = pd.to_numeric(vents_df['depth_m'], errors='coerce').dropna()
        print(f'   Deepest vent (m)   : {depths.max():.0f}')
        print(f'   Shallowest vent (m): {depths.min():.0f}')
        print(f'   Average depth (m)  : {depths.mean():.0f}')

    # Plotly scatter map of vents
    fig = px.scatter_geo(
        vents_df, lat='lat', lon='lon',
        hover_name='name' if 'name' in vents_df.columns else None,
        color='depth_m' if 'depth_m' in vents_df.columns else None,
        color_continuous_scale='Blues_r',
        title='🔵 Global Hydrothermal Vent Fields (InterRidge v3.4)',
        projection='natural earth',
        size_max=5,
    )
    fig.update_layout(
        geo=dict(showocean=True, oceancolor='#0a1a4e',
                 showland=True, landcolor='#2d4a22',
                 bgcolor='#0a0a1e'),
        paper_bgcolor='#0a0a1e', font_color='white',
        height=500
    )
    fig.show()

In [ ]:
if volc_df is not None and len(volc_df) > 10:
    fig2 = px.scatter_geo(
        volc_df, lat='lat', lon='lon',
        hover_name='name',
        title='🌋 Submarine Volcanoes (Smithsonian GVP)',
        projection='natural earth',
        color_discrete_sequence=['#ff6400'],
    )
    fig2.update_traces(marker=dict(size=6, symbol='triangle-up'))
    fig2.update_layout(
        geo=dict(showocean=True, oceancolor='#0a1a4e',
                 showland=True, landcolor='#2d4a22',
                 bgcolor='#0a0a1e'),
        paper_bgcolor='#0a0a1e', font_color='white',
        height=500
    )
    fig2.show()

---
## 🌍 Step 12 — 3D: Interactive Global Ocean Floor Surface

**True interactive 3D surface** using Plotly — rotate, zoom, pan with mouse.

Data: NOAA NCEI `DEM_global_mosaic` ImageServer → float32 elevation raster → Plotly `go.Surface`

> ⏳ **Downloads ~2MB from NOAA** — may take 15–30 seconds on first run

In [ ]:
# Download global bathymetry at coarse resolution for 3D surface
# 360×180 = 1° resolution (fast, smooth global view)
# 720×360 = 0.5° resolution (more detail, ~4x larger download)

try:
    elev_global = fetch_bathymetry(
        bbox=(-180, -90, 180, 90),
        size=(360, 180),
        label='Global (1° resolution)'
    )
    # Replace no-data values
    elev_global = np.where(elev_global < -15000, np.nan, elev_global)
    elev_global = np.where(elev_global > 20000,  np.nan, elev_global)
    # Simple smoothing to reduce noise
    from scipy.ndimage import uniform_filter
    elev_global_smooth = uniform_filter(np.nan_to_num(elev_global, nan=0), size=3)
    print('✅ Data ready for 3D visualization')
except ImportError:
    print('scipy not installed — skipping smoothing (pip install scipy)')
    elev_global_smooth = np.nan_to_num(elev_global, nan=0)
except Exception as e:
    print(f'❌ Download failed: {e}')
    print('   Creating synthetic bathymetry for demo...')
    # Synthetic fallback
    x = np.linspace(-180, 180, 360)
    y = np.linspace(-90, 90, 180)
    xx, yy = np.meshgrid(x, y)
    elev_global_smooth = (-3000
        - 1000 * np.sin(np.radians(xx) * 3)
        + 500  * np.cos(np.radians(yy) * 2)
        + 3500 * np.exp(-((xx/60)**2 + (yy/20)**2))  # ridge
    )
    print('   (Synthetic data — install requests+tifffile for real NOAA data)')

In [ ]:
lons_g = np.linspace(-180, 180, elev_global_smooth.shape[1])
lats_g = np.linspace(-90,  90,  elev_global_smooth.shape[0])

# Mask land (elevation > 0) to focus on ocean floor
ocean_only = elev_global_smooth.copy()
ocean_only[ocean_only > 200] = np.nan   # mask land above +200m

fig_3d = go.Figure(data=[
    go.Surface(
        z           = ocean_only,
        x           = lons_g,
        y           = lats_g,
        colorscale  = OCEAN_COLORSCALE,
        showscale   = True,
        colorbar    = dict(
            title       = 'Elevation (m)',
            tickvals    = [-11000, -8000, -6000, -4000, -2000, -1000, -200, 0],
            ticktext    = ['-11000 (Hadal)', '-8000', '-6000', '-4000',
                           '-2000', '-1000', '-200 (Shelf)', '0 (Sea level)'],
            thickness   = 18,
            len         = 0.8,
        ),
        lighting    = dict(ambient=0.6, diffuse=0.8, specular=0.3, roughness=0.5),
        lightposition = dict(x=100, y=200, z=1000),
        hovertemplate = 'Lon: %{x:.1f}°<br>Lat: %{y:.1f}°<br>Depth: %{z:.0f}m<extra></extra>',
    )
])

fig_3d.update_layout(
    title = dict(
        text='🌊 Global Ocean Floor — Interactive 3D Surface<br>'
             '<sup>Data: NOAA NCEI / GEBCO · Drag to rotate · Scroll to zoom · Right-drag to pan</sup>',
        font=dict(size=16, color='white')
    ),
    scene = dict(
        xaxis = dict(title='Longitude', color='white', gridcolor='#333',
                     backgroundcolor='#0a0a1e', showbackground=True),
        yaxis = dict(title='Latitude',  color='white', gridcolor='#333',
                     backgroundcolor='#0a0a1e', showbackground=True),
        zaxis = dict(title='Elevation (m)', color='white', gridcolor='#333',
                     backgroundcolor='#0a0a1e', showbackground=True,
                     range=[-12000, 3000]),
        camera = dict(
            eye  = dict(x=0, y=-1.8, z=1.2),
            up   = dict(x=0, y=0,   z=1),
        ),
        aspectratio = dict(x=2, y=1, z=0.25),  # flatten z-axis for better viz
        bgcolor     = '#0a0a1e',
    ),
    paper_bgcolor = '#0a0a1e',
    font          = dict(color='white'),
    height        = 700,
    margin        = dict(l=0, r=0, t=80, b=0),
)

fig_3d.show()
print('\n🕹️  3D controls:')
print('   Left-drag   = rotate the surface')
print('   Scroll      = zoom in/out')
print('   Right-drag  = pan')
print('   Hover       = show exact depth at any point')

---
## 🌍 Step 13 — 3D: Regional Ocean Floor Focus
High-resolution 3D view of specific ocean features — trenches, ridges, basins.

**Preset regions:**
| Name | bbox |
|---|---|
| `mariana`     | Mariana Trench (deepest point on Earth, 11,000m) |
| `mid_atlantic`| Mid-Atlantic Ridge (spreading center) |
| `pacific_ring`| Pacific Ring of Fire |
| `indian_ocean`| Indian Ocean |
| `arctic`      | Arctic Ocean |
| `caribbean`   | Caribbean Sea |
| `med`         | Mediterranean Sea |

In [ ]:
# ── Regional bounding boxes ───────────────────────────────────────────────────
REGIONS = {
    'mariana'     : (130,   5,   155,  25,  '🇯🇵 Mariana Trench (deepest: ~11,034m)'),
    'mid_atlantic': (-50,  -60,  -10,  60,  '🌊 Mid-Atlantic Ridge'),
    'pacific_ring': (120,  -50,  -60,  60,  '🌋 Pacific Ring of Fire'),
    'indian_ocean': ( 30,  -60,  120, -10,  '🌊 Indian Ocean'),
    'arctic'      : (-180,  70,  180,  90,  '🧊 Arctic Ocean'),
    'caribbean'   : (-90,    8,  -58,  28,  '🌊 Caribbean Sea'),
    'med'         : ( -6,   30,   42,  46,  '🏛️  Mediterranean Sea'),
}

print('Available regions:')
for key, val in REGIONS.items():
    print(f'   {key:<15}: {val[4]}')

In [ ]:
# ✏️ Change this to any key from REGIONS above:
FOCUS_REGION = 'mariana'

xmin, ymin, xmax, ymax, label = REGIONS[FOCUS_REGION]

# Higher resolution for regional view
try:
    elev_region = fetch_bathymetry(
        bbox=(xmin, ymin, xmax, ymax),
        size=(500, 400),
        label=label
    )
    elev_region = np.where(elev_region < -15000, np.nan, elev_region)
    elev_region = np.where(elev_region > 20000,  np.nan, elev_region)
    try:
        from scipy.ndimage import uniform_filter
        elev_region = uniform_filter(np.nan_to_num(elev_region, nan=0), size=2)
    except ImportError:
        elev_region = np.nan_to_num(elev_region, nan=0)
except Exception as e:
    print(f'Download failed: {e} — using global data cropped to region')
    # Fallback: crop global data
    xi = np.searchsorted(lons_g, [xmin, xmax])
    yi = np.searchsorted(lats_g, [ymin, ymax])
    elev_region = elev_global_smooth[yi[0]:yi[1], xi[0]:xi[1]]

lons_r = np.linspace(xmin, xmax, elev_region.shape[1])
lats_r = np.linspace(ymin, ymax, elev_region.shape[0])

fig_region = go.Figure(data=[
    go.Surface(
        z           = elev_region,
        x           = lons_r,
        y           = lats_r,
        colorscale  = OCEAN_COLORSCALE,
        showscale   = True,
        colorbar    = dict(title='Elevation (m)', thickness=15, len=0.8),
        lighting    = dict(ambient=0.5, diffuse=0.9, specular=0.4, roughness=0.4),
        lightposition = dict(x=500, y=500, z=2000),
        hovertemplate = 'Lon: %{x:.2f}°<br>Lat: %{y:.2f}°<br>Depth: %{z:.0f}m<extra></extra>',
    )
])

fig_region.update_layout(
    title  = dict(
        text=f'🌊 {label} — High-Resolution 3D Ocean Floor<br>'
             '<sup>Drag to rotate · Scroll to zoom · Hover for depth</sup>',
        font=dict(size=15, color='white')
    ),
    scene  = dict(
        xaxis = dict(title='Longitude', color='white', backgroundcolor='#0a0a1e',
                     showbackground=True, gridcolor='#222'),
        yaxis = dict(title='Latitude',  color='white', backgroundcolor='#0a0a1e',
                     showbackground=True, gridcolor='#222'),
        zaxis = dict(title='Elevation (m)', color='white', backgroundcolor='#0a0a1e',
                     showbackground=True, gridcolor='#222'),
        camera      = dict(eye=dict(x=1.2, y=-1.5, z=1.0)),
        aspectratio = dict(x=1.5, y=1, z=0.4),
        bgcolor     = '#0a0a1e',
    ),
    paper_bgcolor = '#0a0a1e',
    font          = dict(color='white'),
    height        = 680,
    margin        = dict(l=0, r=0, t=80, b=0),
)

fig_region.show()

---
## 🌍 Step 14 — 3D: Ocean Floor + Hydrothermal Vents + Volcanoes Overlay
3D surface with vent fields and submarine volcanoes plotted as 3D scatter points.

In [ ]:
traces = [
    # Ocean floor surface
    go.Surface(
        z           = ocean_only,
        x           = lons_g,
        y           = lats_g,
        colorscale  = OCEAN_COLORSCALE,
        showscale   = True,
        opacity     = 0.92,
        colorbar    = dict(title='Depth (m)', thickness=15, len=0.7, x=1.02),
        lighting    = dict(ambient=0.6, diffuse=0.8, roughness=0.5),
        name        = 'Ocean Floor',
        hovertemplate = 'Depth: %{z:.0f}m<extra>Ocean Floor</extra>',
    )
]

# Add hydrothermal vents as 3D scatter
if vents_df is not None:
    depth_z = pd.to_numeric(vents_df.get('depth_m', pd.Series()), errors='coerce')
    vent_z  = (-depth_z.fillna(2000)).tolist()  # place dots at vent depth
    traces.append(go.Scatter3d(
        x    = vents_df['lon'].tolist(),
        y    = vents_df['lat'].tolist(),
        z    = vent_z,
        mode = 'markers',
        name = '🔵 Hydrothermal Vents',
        marker = dict(size=4, color='cyan', opacity=0.9,
                      line=dict(color='white', width=0.5)),
        hovertemplate = (
            '<b>%{text}</b><br>Lon: %{x:.2f}°<br>Lat: %{y:.2f}°'
            '<br>Depth: %{z:.0f}m<extra>Hydrothermal Vent</extra>'
        ),
        text = vents_df.get('name', pd.Series(['Vent']*len(vents_df))).tolist(),
    ))

# Add submarine volcanoes as 3D scatter
if volc_df is not None:
    traces.append(go.Scatter3d(
        x    = volc_df['lon'].tolist(),
        y    = volc_df['lat'].tolist(),
        z    = [0] * len(volc_df),   # place at sea level / summit
        mode = 'markers',
        name = '🌋 Submarine Volcanoes',
        marker = dict(size=5, color='#ff6400', opacity=0.95,
                      symbol='diamond', line=dict(color='white', width=0.5)),
        hovertemplate = (
            '<b>%{text}</b><br>Lon: %{x:.2f}°<br>Lat: %{y:.2f}°'
            '<extra>Submarine Volcano</extra>'
        ),
        text = volc_df['name'].tolist(),
    ))

fig_overlay = go.Figure(data=traces)
fig_overlay.update_layout(
    title  = dict(
        text='🌊 Ocean Floor + Hydrothermal Vents + Submarine Volcanoes<br>'
             '<sup>🔵 Cyan = vents  |  🟠 Orange = volcanoes  |  Hover for details</sup>',
        font=dict(size=15, color='white')
    ),
    scene  = dict(
        xaxis = dict(title='Longitude', color='white', backgroundcolor='#0a0a1e',
                     showbackground=True, gridcolor='#222'),
        yaxis = dict(title='Latitude',  color='white', backgroundcolor='#0a0a1e',
                     showbackground=True, gridcolor='#222'),
        zaxis = dict(title='Elevation (m)', color='white', backgroundcolor='#0a0a1e',
                     showbackground=True, gridcolor='#222', range=[-12000, 2000]),
        camera      = dict(eye=dict(x=0, y=-2.0, z=1.2)),
        aspectratio = dict(x=2, y=1, z=0.2),
        bgcolor     = '#0a0a1e',
    ),
    paper_bgcolor = '#0a0a1e',
    font          = dict(color='white'),
    legend        = dict(bgcolor='rgba(20,20,40,0.8)', font=dict(color='white')),
    height        = 700,
    margin        = dict(l=0, r=0, t=80, b=0),
)
fig_overlay.show()

---
## 🗺️ Step 15 — 2D ArcGIS: Focus on Any Ocean Region
Change `OCEAN_FOCUS` to zoom any 2D map into a specific ocean feature.

In [ ]:
# ✏️ Change to any ocean or region:
# 'Mariana Trench' | 'Mid-Atlantic Ridge' | 'Indian Ocean' |
# 'Red Sea' | 'Arctic Ocean' | 'Mediterranean Sea' | 'Caribbean Sea'
OCEAN_FOCUS = 'Mariana Trench'

res = geocode(OCEAN_FOCUS, max_locations=1)

focus_map = make_map(basemap='oceans', zoom=5, height='600px')

if res:
    r = res[0]
    focus_map.center = [r['location']['y'], r['location']['x']]
    print(f'📍 Centered on: {r["address"]}')

print('\nLoading layers...')
for url, name in [
    (URLS['bathy_hillshade'],  'Bathymetry'),
    (URLS['gebco_contours'],   'GEBCO contours'),
    (URLS['plate_boundaries'], 'Plate boundaries'),
]:
    lyr = safe_load(url, name)
    if lyr: focus_map.content.add(lyr)

# Draw nearby vents
if res and vents_df is not None:
    clat, clon = r['location']['y'], r['location']['x']
    nearby = vents_df[
        (vents_df['lat'].between(clat-25, clat+25)) &
        (vents_df['lon'].between(clon-25, clon+25))
    ]
    if len(nearby) > 0:
        draw_points(focus_map, nearby, 'lat', 'lon',
                    color=[0, 220, 210, 240], size=10, label='nearby vent fields')

focus_map

---
## 📋 Quick Reference

### All Confirmed Working URLs
```python
NOAA = 'https://gis.ngdc.noaa.gov/arcgis/rest/services'

# Raster (ImageServer)
f'{NOAA}/DEM_mosaics/DEM_global_mosaic_hillshade/ImageServer'  # Bathymetry hillshade
f'{NOAA}/DEM_mosaics/DEM_global_mosaic/ImageServer'            # Bathymetry DEM (for 3D)
f'{NOAA}/DEM_mosaics/ETOPO1_bedrock/ImageServer'               # ETOPO1
f'{NOAA}/EMAG2v3/ImageServer'                                  # Magnetic anomaly

# Vector (MapServer)
f'{NOAA}/wgs84/gebco_2019_contours/MapServer'                  # Depth contours
f'{NOAA}/wgs84/multibeam_footprints/MapServer'                 # Sonar surveys
f'{NOAA}/reference/coastlines/MapServer'                       # Coastlines
```

### 3D Surface (core pattern)
```python
# 1. Download elevation grid from NOAA
elev = fetch_bathymetry(bbox=(-180,-90,180,90), size=(360,180))
# 2. Mask land
ocean = np.where(elev > 200, np.nan, elev)
# 3. Plot
fig = go.Figure(go.Surface(z=ocean, x=lons, y=lats,
                            colorscale=OCEAN_COLORSCALE))
fig.show()
```

### Scene() vs Map() rule
```
Scene.content.add(MapImageLayer)  ❌ crashes
Scene.content.draw()              ❌ method does not exist
Map.content.add(MapImageLayer)    ✅ works
Map.content.draw()                ✅ works
Plotly go.Surface                 ✅ true 3D interactive surface
```

### 3D Presets
```python
# Change FOCUS_REGION to any key:
'mariana' | 'mid_atlantic' | 'pacific_ring' | 'indian_ocean' | 'arctic' | 'caribbean' | 'med'
```